In [111]:
gen_report = False

In [112]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})


df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 4


In [113]:
def filter_df(filters, df=df):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=["seed", "freeze_object_encoder", "reset_unfrozen_params", "freeze_codebook", "agent_a_training_mode", "freeze_agent_b", "sampling_temperature", "commitment_weight", "entropy_regularization_factor", "learning_rate_phase1", "learning_rate_phase2_b", "learning_rate_phase2_a"])
    

In [114]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    if 'mutual_play_accuracy' in df:
        max_col = 'mutual_play_accuracy'
    else:
        max_col = 'test_accuracy'
    max_val = pd.to_numeric(df[max_col]).max()

    def highlight_max_row(row):
        if  pd.to_numeric(row[max_col]) == max_val:
            return ['font-weight: bold; background-color: #ffff99'] * len(row)
        else:
            return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01 else x)
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#4b0082",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [115]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [116]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "agent_a_training_mode"], max_col="mutual_play_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[section["mutual_play_accuracy"].idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=["agent_a_training_mode"])


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [117]:
clear()

---

# EXP1: VQEL (3 Modes) vs. Baseline

In [118]:
add_heading(1, "EXP1: VQEL (3 Modes) vs. VQ-Reinforce vs. Baseline")

write(
"""
<b>VQEL - Euclidean or Cosine</b>
Here, we train VQEL in the self-play phase for 50 epochs, followed by another 50 epochs in mutual play.
Mutual play has three modes that determine the training behavior of the sender:

- Frozen: the sender's parameters are fixed.
- Reinforce only: the sender is fine tuned using the reinforce algorithm.
- Reinforce with preservation: the sender is fine tuned using both the reinforce loss and the self-play loss.

We also use two different codebooks, depending on the distance metric used to retrieve the nearest vector:
- Euclidean distance
- Cosine similarity

<b>VQ-Reinforce</b>
The architecture of VQ-Reinforce is the same as VQEL, but it is trained for 100 epochs using REINFORCE, without any self-play.

<b>Baseline</b>
The baseline is trained for 100 epochs.

"""
)

In [119]:
vq_cols = [
    "seed",
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "agent_a_training_mode", 
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "commitment_weight",
    "sampling_temperature",
    "representation_dim",
    "pretrained_checkpoint_a",
    "path",
]

vq_rf_cols = [
    "seed",
    "VQEL",
    "dataset",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "mutual_play_accuracy",
    "commitment_weight",
    "sampling_temperature",
    "representation_dim",
    "agent_a_training_mode",
    "num_pretrain_epochs",
    "contrastive_loss_temperature",
    "path"
]

bs_cols = [
    "VQEL",
    "dataset",
    "contrastive_loss_temperature",
    "entropy_regularization_factor",
    "learning_rate",
    "test_accuracy",
    "sampling_temperature",
    "representation_dim",
    "path"
]

vq_rf_cols_report = [
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "commitment_weight",
    "sampling_temperature",
    "mutual_play_accuracy",
]

vq_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

bs_cols_report = [
    "learning_rate",
    "test_accuracy",
]

## Objects

### VQEL - Euclidean

In [120]:
add_heading(2, "Objects")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "objects",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path


### VQEL - Cosine

In [121]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "objects",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

res[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path


### VQ-Reinforce

In [122]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "objects",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

,seed,VQEL,dataset,learning_rate_phase2_a,learning_rate_phase2_b,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,agent_a_training_mode,num_pretrain_epochs,contrastive_loss_temperature,path


### Baseline

In [123]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "objects",
    "VQEL": False,
})


## ShapeWorld

### VQEL - Euclidean

In [124]:
add_heading(2, "ShapeWorld")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path


### VQEL - Cosine

In [125]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
1,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-03,1e-03,1e-03,0.754,0.817,0.25,1e-05,1024,None,20251219_1339_bs32_vocab10_repr1024_lr1_0.001_...
0,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-03,1e-04,1e-04,0.754,0.832,0.25,1e-05,1024,None,20251219_1426_bs32_vocab10_repr1024_lr1_0.001_...
2,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-04,1e-04,1e-04,0.684,0.838,0.25,1e-05,1024,None,20251219_1352_bs32_vocab10_repr1024_lr1_0.0001...
3,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-05,1e-05,1e-05,0.676,0.801,0.25,1e-05,1024,None,20251219_1406_bs32_vocab10_repr1024_lr1_1e-05_...


### VQ-Reinforce

In [126]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "shape",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

,seed,VQEL,dataset,learning_rate_phase2_a,learning_rate_phase2_b,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,agent_a_training_mode,num_pretrain_epochs,contrastive_loss_temperature,path


### Baseline

In [127]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape",
    "VQEL": False,
    "representation_dim": 1024,
    "seed": 1
})

to_html(res[bs_cols_report])

res[bs_cols]

KeyError: "None of [Index(['learning_rate', 'test_accuracy'], dtype='object')] are in the [columns]"

## DSprites

### VQEL - Euclidean

In [ ]:
add_heading(2, "DSprites")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "dsprites",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

### VQEL - Cosine

In [ ]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "dsprites",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

to_html(res[vq_cols_report])

res[vq_cols]

### VQ-Reinforce

In [ ]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "dsprites",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

### Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "dsprites",
    "VQEL": False,
})

to_html(res[bs_cols_report])

res[bs_cols]

## CelebA

### VQEL - Euclidean

In [ ]:
add_heading(2, "CelebA")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "celeba",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

### VQEL - Cosine

In [ ]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "celeba",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

to_html(res[vq_cols_report])

res[vq_cols]

### VQ-Reinforce

In [ ]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "celeba",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

### Baseline

In [ ]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "celeba",
    "VQEL": False,
}).sort_values(by=["contrastive_loss_temperature", "entropy_regularization_factor", "learning_rate"])

to_html(res[bs_cols_report])

res[bs_cols]